# B2.5 · Sub-agents and delegation depth

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *Security of AI*

Builds on **[B2.4 · Model tiering and routing inside the loop](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**.

| | |
|---|---|
| Open-source tooling | kagent, SPIRE |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

A sub-agent is a second loop with its own context and its own share of your privileges. Sometimes that halves the problem. Sometimes it multiplies the error rate and the blast radius at the same time, three levels deep.

> **At CyberTravels.** A sub-agent reviewing the payments service is a second loop holding a share of CyberTravels' repository credentials. Sometimes that halves the problem; sometimes it multiplies the blast radius three levels down.

## 2 · The framework

```
   depth 0   orchestrator          budget 100%   privileges P
   depth 1     +-- sub-agent       budget  40%   privileges <= P
   depth 2         +-- sub-agent   budget  15%   privileges <= P
   depth 3             +-- ???     budget   ?    privileges   ?

   error compounds down the tree. so does authority, unless it narrows.
```

Sub-agents buy specialisation. What they also buy — and what is never on the
roadmap — is **delegation depth**.

Two things grow with depth, and only one of them is obvious:

- **Authority composition** (A1.3, A2.5). Each hop is a place authority could
  widen if the narrowing rules are not enforced at *every* hop.
- **Attribution distance.** By hop four, the action is five identities away from
  the human who asked, and no single reviewer has seen the whole path.

The control is a depth limit, and the interesting question is **where to enforce
it**. Enforcing it in the orchestrator is the obvious choice and the wrong one:
when the orchestrator is the compromised component, its own check is worth
nothing. It has to live at the token issuer or the resource server.

## 3 · Demo — a depth-3 sub-agent chain that narrows correctly

In [ ]:
from dataclasses import dataclass, field

CEILINGS = {"dana@corp": {"repo:read","repo:write","deploy:prod"},
            "orchestrator": {"repo:read","repo:write","deploy:prod"},
            "planner":   {"repo:read"},
            "coder":     {"repo:read","repo:write"},
            "reviewer":  {"repo:read"},
            "shipper":   {"repo:read","deploy:prod"}}

class DelegationError(Exception): pass

@dataclass
class Token:
    sub: str; actor: str; scopes: set; act: dict = None
    def chain(self):
        out, node = [], self.act
        while node: out.append(node["actor"]); node = node.get("act")
        c = list(reversed(out)) + [self.actor]
        if c[0] != self.sub: c.insert(0, self.sub)
        return c
    @property
    def depth(self): return len(self.chain())

def exchange(pres, actor, scopes):
    scopes = set(scopes)
    if not scopes <= pres.scopes:
        raise DelegationError(f"widening: {sorted(scopes - pres.scopes)}")
    if not scopes <= CEILINGS.get(actor, set()):
        raise DelegationError(f"above {actor}'s ceiling")
    return Token(pres.sub, actor, scopes, {"actor": pres.actor, "act": pres.act})

root  = Token("dana@corp", "dana@corp", set(CEILINGS["dana@corp"]))
orch  = exchange(root, "orchestrator", {"repo:read","repo:write","deploy:prod"})
coder = exchange(orch, "coder", {"repo:read","repo:write"})
rev   = exchange(coder, "reviewer", {"repo:read"})

for t in (root, orch, coder, rev):
    print(f"depth {t.depth}  {' → '.join(t.chain()):52s} {sorted(t.scopes)}")

## 4 · Where it breaks — the check in the wrong place

The orchestrator enforces `MAX_DEPTH`. Now assume the orchestrator is the compromised component, which is the realistic case: it is the one processing untrusted task descriptions.

In [ ]:
MAX_DEPTH = 3

def orchestrator_enforced(chain_token, new_actor, scopes, honest=True):
    """The limit lives inside the orchestrator's own code."""
    if honest and chain_token.depth + 1 > MAX_DEPTH:
        raise DelegationError(f"depth {chain_token.depth+1} > {MAX_DEPTH}")
    return exchange(chain_token, new_actor, scopes)

print("honest orchestrator:")
try:
    t = orchestrator_enforced(rev, "shipper", {"repo:read"}, honest=True)
    print("   granted:", t.chain())
except DelegationError as e:
    print("   refused:", e)

print("\ncompromised orchestrator (simply does not run its own check):")
t = orchestrator_enforced(rev, "shipper", {"repo:read"}, honest=False)
print(f"   granted: depth {t.depth}  {' → '.join(t.chain())}")
print("   The limit was a line of code inside the component we no longer trust.")

## 5 · The control — enforce at the issuer and the resource server

Both of these are outside the orchestrator's control, so a compromised orchestrator cannot skip them.

In [ ]:
def issuer_exchange(pres, actor, scopes, max_depth=MAX_DEPTH):
    """The token issuer counts the act chain it is being asked to extend."""
    if pres.depth + 1 > max_depth:
        raise DelegationError(f"issuer refuses: depth {pres.depth+1} > {max_depth}")
    return exchange(pres, actor, scopes)

def resource_server(token, scope, max_depth=MAX_DEPTH):
    """Independent second check, at the point the action actually happens."""
    if token.depth > max_depth:
        return False, f"resource server refuses: depth {token.depth} > {max_depth}"
    if scope not in token.scopes:
        return False, f"missing scope {scope}"
    return True, f"allowed for {' → '.join(token.chain())}"

print("issuer enforcement:")
try:
    issuer_exchange(rev, "shipper", {"repo:read"})
except DelegationError as e:
    print("   ", e)

print("\nresource server enforcement (even if a deep token somehow exists):")
deep = Token("dana@corp", "shipper", {"repo:read"},
             {"actor":"reviewer","act":{"actor":"coder",
              "act":{"actor":"orchestrator","act":None}}})
print(f"   depth {deep.depth}: {resource_server(deep, 'repo:read')}")
print(f"   depth {rev.depth}: {resource_server(rev, 'repo:read')}")

In [ ]:
# Verify: no chain the compromised orchestrator can build is ever honoured.
import random
random.seed(9)
actors = ["planner","coder","reviewer","shipper"]
honoured_too_deep = 0
for _ in range(3000):
    tok = Token("dana@corp","dana@corp", set(CEILINGS["dana@corp"]))
    for _ in range(random.randint(1,6)):
        a = random.choice(actors)
        want = set(random.sample(sorted(tok.scopes),
                                 k=random.randint(0,len(tok.scopes))))
        try:
            tok = exchange(tok, a, want)      # compromised: no depth check
        except DelegationError:
            break
    ok, _ = resource_server(tok, "repo:read")
    if ok and tok.depth > MAX_DEPTH:
        honoured_too_deep += 1
print(f"3000 chains built without any orchestrator-side limit — "
      f"over-deep chains honoured by the resource server: {honoured_too_deep}")
assert honoured_too_deep == 0
print("The limit holds because it is enforced where the orchestrator cannot reach.")

## What you just proved

The three-hop chain narrows correctly to `dana@corp → orchestrator → coder → reviewer`. An honest orchestrator refuses the fourth hop; a compromised one grants it. The issuer refuses the same exchange, and the resource server refuses a depth-4 token while allowing depth 3. Over 3,000 randomly-built chains, zero over-deep chains are honoured.

## Your turn

Draw your own agent call graph and count the longest path. Then find where the depth limit is enforced. If the answer is "in the orchestrator", move it — the component that builds the chain cannot be the one that bounds it.

---

**Next → [B2.6 · Failure taxonomy](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*